# WIOD parser walkthrough

This notebook is the practical guide for parsing WIOD 2016 workbooks in MARIO.


## What this notebook covers

- where to download each WIOD workbook family;
- when to pass a direct file path and when to pass a directory;
- how `year=` and `country=` disambiguate candidate files;
- how `MRIO`, national `IOT`, and national `SUT` workflows differ;
- how `add_extensions=` imports `Socio_Economic_Accounts.xlsx`;
- why the international `SUT` parser exposes `row_mode=` and why that part should still be treated with caution.


## Relevant source pages

- Official WIOD 2016 release page: [GGDC WIOD 2016 release](https://www.rug.nl/ggdc/valuechain/wiod/wiod-2016-release?lang=en)
- MRIO IOT, current prices: [Dataverse 199104](https://dataverse.nl/api/access/datafile/199104)
- MRIO IOT, previous-year prices: [Dataverse 199102](https://dataverse.nl/api/access/datafile/199102)
- MRIO international SUT: [Dataverse 199100](https://dataverse.nl/api/access/datafile/199100)
- National IOT bundle: [Dataverse 199099](https://dataverse.nl/api/access/datafile/199099)
- National SUT bundle: [Dataverse 199096](https://dataverse.nl/api/access/datafile/199096)
- Socio-economic accounts: [Dataverse 199095](https://dataverse.nl/api/access/datafile/199095)


## Main entry point

For normal user workflows, the public entry point is:

- `mario.parse_wiod(...)`

The same function supports:

- multiregional `IOT` workbooks;
- multiregional `IOT` `_PYP` workbooks;
- multiregional international `SUT` workbooks;
- national `IOT` workbooks;
- national `SUT` workbooks;
- optional socio-economic extensions through `add_extensions=`.


## Key arguments

The key public arguments are:

- `path`: one workbook or one directory containing multiple WIOD workbooks;
- `table`: choose `"IOT"` or `"SUT"`;
- `year`: used to disambiguate directories and mandatory for national workbooks with multi-year content;
- `country`: useful when one directory contains multiple national workbooks;
- `add_extensions`: optional path to `Socio_Economic_Accounts.xlsx`;
- `row_mode`: only relevant for the international `SUT` workbook.


## Direct file path vs directory path

Use a **direct workbook path** when you already know the exact file to parse. This is the simplest case and often means you do not need `year=` or `country=`.

Use a **directory path** when you keep several WIOD files together. In that case MARIO scans the directory and then uses `table=`, `year=`, and optionally `country=` to pick the correct file.


## Example local layouts and caveats

Typical extracted layout:

```text
WIOD/
├── WIOTS_in_EXCEL/
│   └── WIOT2014_Nov16_ROW.xlsb
├── WIOTS_PYP_in_EXCEL/
│   └── WIOT2014_PYP_Nov16_ROW.xlsb
├── NIOTS/
│   └── ITA_NIOT_nov16.xlsx
├── SUT_national/
│   └── ITA_SUT_nov16.xlsx
├── SUT_international/
│   └── intsut14_nov16.xlsb
└── Socio_Economic_Accounts.xlsx
```

National `IOT` and `SUT` workbooks contain multiple years in one file, so `year=` is mandatory for those inputs. For the international `SUT`, `row_mode="external_account"` is the recommended mode because `ROW` is not published as a complete endogenous economy.

In [1]:
import mario

## Optional download step

MARIO provides dedicated WIOD download helpers instead of one generic downloader for every asset.


In [ ]:
mario.download_wiod2016(
    path="/path/to/wiod",
    table="IOT",
)

mario.download_wiod2016_iot_pyp("/path/to/wiod_pyp")
mario.download_wiod2016_national_iot("/path/to/wiod_national_iot")
mario.download_wiod2016_national_sut("/path/to/wiod_national_sut")
mario.download_wiod2016_socioeconomic_accounts("/path/to/wiod_extensions")


## Parse one explicit MRIO IOT workbook

Use this when you want to point MARIO to one specific WIOD multiregional IOT file.


In [2]:
db = mario.parse_wiod(
    path="/path/to/WIOT2014_Nov16_ROW.xlsb",
    table="IOT",
)


INFO Parser: reading WIOD workbook WIOT2014_Nov16_ROW.xlsb sheet 2014.


INFO Parser: WIOD IOT payload ready with shapes Z=(2464, 2464), Y=(2464, 220), V=(6, 2464).


INFO Metadata: initialized.


## Parse one explicit MRIO IOT workbook in previous-year prices

The `_PYP` variant is selected from the filename directly. No extra parser flag is needed.


In [3]:
db = mario.parse_wiod(
    path="/path/to/WIOT2014_PYP_Nov16_ROW.xlsb",
    table="IOT",
)


INFO Parser: reading WIOD workbook WIOT2014_PYP_Nov16_ROW.xlsb sheet 2014.


INFO Parser: WIOD IOT payload ready with shapes Z=(2464, 2464), Y=(2464, 220), V=(6, 2464).


INFO Metadata: initialized.


## Parse one international SUT workbook

This is the source where the treatment of `ROW` matters.


In [4]:
db = mario.parse_wiod(
    path="/path/to/intsut14_nov16.xlsb",
    table="SUT",
    row_mode="external_account",
)


INFO Parser: reading WIOD workbook intsut14_nov16.xlsb sheet SUP.


INFO Parser: reading WIOD workbook intsut14_nov16.xlsb sheet USE.


INFO Parser: WIOD SUT payload ready with shapes S=(2408, 2408), U=(2408, 2408), Yc=(2408, 258), Va=(62, 2408), VY=(62, 258).


INFO Metadata: initialized.


## Parse one national IOT workbook

National WIOD `IOT` workbooks contain multiple years in one workbook, so `year=` is required.


In [5]:
db = mario.parse_wiod(
    path="/path/to/ITA_NIOT_nov16.xlsx",
    table="IOT",
    year=2014,
)


INFO Parser: reading WIOD workbook ITA_NIOT_nov16.xlsx sheet National IO-tables.


INFO Metadata: initialized.


## Parse one national SUT workbook

National WIOD `SUT` workbooks also require `year=`.


In [6]:
db = mario.parse_wiod(
    path="/path/to/ITA_SUT_nov16.xlsx",
    table="SUT",
    year=2014,
)


INFO Parser: reading WIOD workbook ITA_SUT_nov16.xlsx sheet SUP.


INFO Parser: reading WIOD workbook ITA_SUT_nov16.xlsx sheet USE.


INFO Metadata: initialized.


## Parse from a directory containing multiple WIOD files

This is the important case when you keep several WIOD files together. Use `year=` to select the release and `country=` when the directory contains multiple national workbooks.


In [7]:
db = mario.parse_wiod(
    path="/path/to/wiod_directory",
    table="SUT",
    country="ITA",
    year=2014,
)


INFO Parser: reading WIOD workbook ITA_SUT_nov16.xlsx sheet SUP.


INFO Parser: reading WIOD workbook ITA_SUT_nov16.xlsx sheet USE.


INFO Metadata: initialized.


## Add socio-economic extensions

When `add_extensions=` is provided, MARIO imports `Socio_Economic_Accounts.xlsx` as satellite extensions.

- for `IOT`, they populate `E`;
- for `SUT`, they populate `Ea` and keep `Ec` zero-filled.


In [8]:
db = mario.parse_wiod(
    path="/path/to/WIOT2014_Nov16_ROW.xlsb",
    table="IOT",
    add_extensions="/path/to/Socio_Economic_Accounts.xlsx",
)


INFO Parser: reading WIOD workbook WIOT2014_Nov16_ROW.xlsb sheet 2014.


INFO Parser: WIOD IOT payload ready with shapes Z=(2464, 2464), Y=(2464, 220), V=(6, 2464).


INFO Metadata: initialized.


## Warnings to expect

Typical parser warnings or important notes are:

- unreadable `.xlsb` files stored as cloud placeholders instead of real local files;
- ambiguous directory selections when `year=` or `country=` is missing;
- notes in `db.meta_history` when the international `SUT` parser reclassifies `ROW` under `row_mode="external_account"`.
